<a href="https://colab.research.google.com/github/KhusbuBubna123/Movie_Sentiment_Analysis1/blob/main/MovieSentimentAnalysis_TF_IDF_Bernoilli_NaiiveBayes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
import string
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB, BernoulliNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer

In [28]:
df = pd.read_csv("IMDBDataset.csv",names=['Movie Review','Sentiment'])
df.head(10)

,Movie Review,Sentiment
0,review,sentiment
1,One of the other reviewers has mentioned that ...,positive
2,A wonderful little production. <br /><br />The...,positive
3,I thought this was a wonderful way to spend ti...,positive
4,Basically there's a family where a little boy ...,negative
5,"Petter Mattei's ""Love in the Time of Money"" is...",positive
6,"Probably my all-time favorite movie, a story o...",positive
7,I sure would like to see a resurrection of a u...,positive
8,"This show was an amazing, fresh & innovative i...",negative
9,Encouraged by the positive comments about this...,negative


In [29]:
df = df.drop(df.index[0]) #Dropping first row of dataframe.

In [30]:
df.shape

(50000, 2)

In [31]:
df['Sentiment'].value_counts()

,count
Sentiment,
positive,25000
negative,25000


In [32]:
df['Review_Length'] = df['Movie Review'].apply(len)
df.head()

,Movie Review,Sentiment,Review_Length
1,One of the other reviewers has mentioned that ...,positive,1761
2,A wonderful little production. <br /><br />The...,positive,998
3,I thought this was a wonderful way to spend ti...,positive,926
4,Basically there's a family where a little boy ...,negative,748
5,"Petter Mattei's ""Love in the Time of Money"" is...",positive,1317


In [33]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['Sentiment_Encoded'] = le.fit_transform(df['Sentiment'])

print(df.head(7))

                                        Movie Review Sentiment  Review_Length  \
1  One of the other reviewers has mentioned that ...  positive           1761   
2  A wonderful little production. <br /><br />The...  positive            998   
3  I thought this was a wonderful way to spend ti...  positive            926   
4  Basically there's a family where a little boy ...  negative            748   
5  Petter Mattei's "Love in the Time of Money" is...  positive           1317   
6  Probably my all-time favorite movie, a story o...  positive            656   
7  I sure would like to see a resurrection of a u...  positive            726   

   Sentiment_Encoded  
1                  1  
2                  1  
3                  1  
4                  0  
5                  1  
6                  1  
7                  1  


#Text Preprocessing.

In [35]:
#Reving <br> tags
df['Movie Review'] = df['Movie Review'].str.replace('<br />', '', regex=False)

In [37]:
# Removing punctuation and converting to lower case.

def remove_punct(text):
    return ''.join(ch for ch in text if ch not in string.punctuation)

def convert_lowercase(text):
  return text.lower()

df['Review_clean'] = df['Movie Review'].apply(remove_punct)
df['Review_clean'] = df['Review_clean'].apply(convert_lowercase)
df[['Movie Review', 'Review_clean']].head(3)

,Movie Review,Review_clean
1,One of the other reviewers has mentioned that ...,one of the other reviewers has mentioned that ...
2,A wonderful little production. The filming tec...,a wonderful little production the filming tech...
3,I thought this was a wonderful way to spend ti...,i thought this was a wonderful way to spend ti...


#Splitting into Train and Test Data

In [39]:
X = df['Review_clean'].values
Y = df['Sentiment_Encoded'].values.astype(int)

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.20, random_state=42)

In [40]:
print('Train size:', len(X_train))
print('Test  size:', len(X_test))

Train size: 40000
Test  size: 10000


In [41]:
# Select only the first 100 samples for faster training
subset_size = 100
X_train_reduced = X_train[:subset_size]
Y_train_reduced = Y_train[:subset_size]

In [42]:
# Select only the first 50 samples for faster testing
subset_size = 50
X_test_reduced = X_test[:subset_size]
Y_test_reduced = Y_test[:subset_size]

#TF-IDF vectorisation + BernoulliNB

In [47]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
tfidf= TfidfVectorizer()

X_train_reduced = tfidf.fit_transform(X_train_reduced)
X_test_reduced = tfidf.transform(X_test_reduced)

print('TF-IDF features:',len(tf_idf.get_feature_names_out()))
print('Train shape     :',X_train.shape)
print('Test  shape     :',X_test.shape)

TF-IDF features: 4667
Train shape     : (100, 4667)
Test  shape     : (50, 4667)


# Model creation — BernoulliNB

In [53]:
nb = BernoulliNB(alpha=0.01) #alpha is used for Laplace smooothing.
nb.fit(X_train_reduced,Y_train_reduced)


BernoulliNB(alpha=0.01)

In [54]:
Y_Pred = nb.predict(X_test_reduced)
Y_Pred

array([0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0,
       0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0,
       1, 1, 0, 1, 1, 1])

In [55]:
#1-Positive    0-Negative
original_categories = le.inverse_transform(Y_Pred)
print(original_categories)

['negative' 'positive' 'negative' 'positive' 'negative' 'negative'
 'positive' 'negative' 'negative' 'negative' 'negative' 'positive'
 'negative' 'positive' 'negative' 'positive' 'positive' 'positive'
 'negative' 'negative' 'positive' 'negative' 'negative' 'negative'
 'negative' 'negative' 'positive' 'positive' 'negative' 'negative'
 'positive' 'positive' 'negative' 'positive' 'negative' 'negative'
 'negative' 'positive' 'negative' 'negative' 'negative' 'positive'
 'negative' 'negative' 'positive' 'positive' 'negative' 'positive'
 'positive' 'positive']


## Accuracy

In [57]:
print(f'BernoulliNB Accuracy: {accuracy_score(Y_test_reduced,Y_Pred)*100:.2f}%')

BernoulliNB Accuracy: 66.00%


In [58]:
print(classification_report(Y_test_reduced,Y_Pred, target_names=['Positve Sentiment','Negative Sentiment']))

                    precision    recall  f1-score   support

 Positve Sentiment       0.55      0.80      0.65        20
Negative Sentiment       0.81      0.57      0.67        30

          accuracy                           0.66        50
         macro avg       0.68      0.68      0.66        50
      weighted avg       0.71      0.66      0.66        50



In [59]:
ConfusionMatrix = pd.DataFrame(
    confusion_matrix(Y_test_reduced,Y_Pred),
    index=['Actual review positve', 'Actual review negative'],
    columns=['Predicted review positve', 'Predicted review negative']
)

# Display the DataFrame
ConfusionMatrix

,Predicted review positve,Predicted review negative
Actual review positve,16,4
Actual review negative,13,17


#Testing on a New Movie Review.

In [70]:
# Removing punctuation and convert to lower case functions.

def remove_punct(text):
    return ''.join(ch for ch in text if ch not in string.punctuation)

def convert_lowercase(text):
  return text.lower()

In [108]:
#1-Positive    0-Negative
def classify_message(movie_review):
    movie_review = movie_review.replace("<br />", "") #Removing <BR> Tags.
    cleaned  = remove_punct(movie_review) #Removing Punctuations.
    cleaned = convert_lowercase(cleaned) #Converting to loercase.

    features = tfidf.transform([cleaned])
    Pred = nb.predict(features)
    if(Pred == 1):
        Sentiment='Positive Sentiment'
    else:
      Sentiment='Negative Sentiment'
    return Sentiment

In [109]:
MovieReview1='''Petter Mattei's "Love in the Time of Money" is a visually stunning film to watch.
Mr. Mattei offers us a vivid portrait about human relations. This is a movie that seems to be telling
us what money, power and success do to people in the different situations we encounter. <br /><br />
This being a variation on the Arthur Schnitzler's play about the same theme, the director transfers
the action to the present time New York where all these different characters meet and connect.
Each one is connected in one way, or another to the next person, but no one seems to know the
previous point of contact. Stylishly, the film has a sophisticated luxurious look. We are taken
to see how these people live and the world they live in their own habitat.<br /><br />The only
thing one gets out of all these souls in the picture is the different stages of loneliness each
one inhabits. A big city is not exactly the best place in which human relations find sincere
fulfillment, as one discerns is the case with most of the people we encounter.<br /><br />The
acting is good under Mr. Mattei's direction. Steve Buscemi, Rosario Dawson, Carol Kane, Michael
Imperioli, Adrian Grenier, and the rest of the talented cast, make these characters come alive.
<br /><br />We wish Mr. Mattei good luck and await anxiously for his next work.'''

In [110]:
classify_message(MovieReview1)

'Positive Sentiment'

In [111]:
MovieReview2='''A wonderful little production. <br /><br />The filming technique is very unassuming- very old-time-BBC fashion and gives a comforting,
 and sometimes discomforting, sense of realism to the entire piece. <br /><br />The actors are extremely well chosen- Michael Sheen not only "has got
  all the polari" but he has all the voices down pat too! You can truly see the seamless editing guided by the references to Williams' diary entries,
  not only is it well worth the watching but it is a terrificly written and performed piece. A masterful production about one of the great master's of
  comedy and his life. <br /><br />The realism really comes home with the little things: the fantasy of the guard which, rather than use the traditional
  'dream' techniques remains solid then disappears. It plays on our knowledge and our senses, particularly with the scenes concerning Orton and Halliwell
  and the sets (particularly of their flat with Halliwell's murals decorating every surface) are terribly well done.'''

In [112]:
classify_message(MovieReview2)

'Positive Sentiment'

In [113]:
MovieReview3='''Basically there's a family where a little boy (Jake) thinks there's a zombie
in his closet & his parents are fighting all the time.<br /><br />This movie is slower than a
soap opera... and suddenly, Jake decides to become Rambo and kill the zombie.<br /><br />OK,
first of all when you're going to make a film you must Decide if its a thriller or a drama!
As a drama the movie is watchable. Parents are divorcing & arguing like in real life. And then
we have Jake with his closet which totally ruins all the film! I expected to see a BOOGEYMAN
similar movie, and instead i watched a drama with some meaningless thriller spots.<br /><br />
3 out of 10 just for the well playing parents & descent dialogs. As for the shots with Jake:
just ignore them.'''

In [114]:
classify_message(MovieReview3)

'Negative Sentiment'

In [116]:
MovieReview4='''This show was an amazing, fresh & innovative idea in the 70's when it first aired.
The first 7 or 8 years were brilliant, but things dropped off after that. By 1990, the show was not
really funny anymore, and it's continued its decline further to the complete waste of time it is today.
<br /><br />It's truly disgraceful how far this show has fallen. The writing is painfully bad, the
performances are almost as bad - if not for the mildly entertaining respite of the guest-hosts, this
show probably wouldn't still be on the air. I find it so hard to believe that the same creator that
hand-selected the original cast also chose the band of hacks that followed. How can one recognize such
brilliance and then see fit to replace it with such mediocrity? I felt I must give 2 stars out of respect
for the original cast that made this show such a huge success. As it is now, the show is just awful.
I can't believe it's still on the air.'''

In [117]:
classify_message(MovieReview4)

'Negative Sentiment'